In [46]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from core.signal.preprocess import *
from glob import glob
import torchaudio
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import *
import os
import torchinfo
from core.nn.basic_dataloader import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
# vctkds = torchaudio.datasets.VCTK_092(root="vctk/", download=False)
# maxLen = max([e[0].shape[1] for e in vctkds])
# for fn in tqdm(collectedFLAC):
#     x, sr = torchaudio.load(fn)
#     tform = torchaudio.transforms.Resample(sr, Fs)
#     x = tform(x)
#     # pad to maxLen
#     x = F.pad(x, (0, maxLen - x.shape[1]))
#     newFn = fn.replace("vctk/VCTK-Corpus-0.92", "vctk/resampled/VCTK-Corpus-0.92")
#     # make sure the directory exists
#     os.makedirs(os.path.dirname(newFn), exist_ok=True)
#     torchaudio.save(newFn, x, Fs)

  0%|          | 0/8942 [00:00<?, ?it/s]

100%|██████████| 8942/8942 [01:06<00:00, 133.65it/s]


In [71]:
vctkds = torchaudio.datasets.VCTK_092(root="vctk/resampled/", download=False)
# Split into train and test
split = 0.85
lenTrain = int(len(vctkds) * split)
lenTest = len(vctkds) - lenTrain
# select lenTrain random indices
np.random.seed(0) # for reproducibility
idxtrain = np.random.choice(len(vctkds), lenTrain, replace=False)
idxtest = np.setdiff1d(np.arange(len(vctkds)), idxtrain)
train = torch.utils.data.Subset(vctkds, idxtrain)
test = torch.utils.data.Subset(vctkds, idxtest)
train_loader = torch.utils.data.DataLoader(train, batch_size=16, shuffle=True)
test_loader = torch.utils.data.DataLoader(test, batch_size=16, shuffle=False)

In [84]:
ds2 = VCTK_092("vctk/gen/csv")
trainds2 = torch.utils.data.Subset(ds2, idxtrain)
testds2 = torch.utils.data.Subset(ds2, idxtest)
train_ds2_loader = torch.utils.data.DataLoader(trainds2, batch_size=16, shuffle=True)
test_ds2_loader = torch.utils.data.DataLoader(testds2, batch_size=16, shuffle=False)

In [76]:
speaker_list = ['p225', 'p226','p227','p228', 'p229', 'p230', 'p231', 'p232', 'p233', 'p234', 'p236', 'p237', 'p238', 'p239', 'p240', 'p241', 'p243', 'p244', 'p245', 'p246', 'p247', 'p248', 'p249']
nSpeakers = len(speaker_list)
onehot_speaker = lambda x: torch.eye(nSpeakers)[speaker_list.index(x)]
# Is this a clean way to do this? Hell nah
# Is this efficient? Yes

In [77]:
for (waveform, _, _, speaker_id, _) in train_loader:
    speaker_one_hot = (torch.stack([onehot_speaker(i) for i in speaker_id]))
    print(waveform.shape)
    break

torch.Size([16, 1, 226209])


In [78]:
class ConvNet2D(nn.Module):
    def __init__(self, nSpeakers):
        super(ConvNet2D, self).__init__()
        self.spec = torchaudio.transforms.Spectrogram(n_fft=1024, hop_length=256, win_length=1024, power=1)
        self.conv1 = nn.Conv2d(2, 16, kernel_size=(9, 9), stride=(2, 2))
        self.pool1 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        self.conv2 = nn.Conv2d(16, 32, kernel_size=(7, 7), stride=(1, 1))
        self.pool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        self.conv3 = nn.Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1))
        self.pool3 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        self.conv4 = nn.Conv2d(64, 128, kernel_size=(5, 5), stride=(1, 1))
        self.pool4 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        self.conv5 = nn.Conv2d(128, 128, kernel_size=(5, 5), stride=(2, 2))
        self.pool5 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        
        self.fc1 = nn.Linear(1280, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc_final = nn.Linear(128, nSpeakers)
    
    def forward(self, x):
        x = self.spec(x)
        
        x = self.conv1(x)
        x = F.relu(x)
        x = self.pool1(x)
        
        x = self.conv2(x)
        x = F.relu(x)
        x = self.pool2(x)
        
        x = self.conv3(x)
        x = F.relu(x)
        x = self.pool3(x)
        
        x = self.conv4(x)
        x = F.relu(x)
        x = self.pool4(x)
        
        x = self.conv5(x)
        x = F.relu(x)
        x = self.pool5(x)
        
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = F.relu(x)
        x = self.fc2(x)
        x = F.relu(x)
        x = self.fc_final(x)
        return x


In [80]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
net = ConvNet2D(nSpeakers).to(device)
torchinfo.summary(net, (1, 2, 226209))

Layer (type:depth-idx)                   Output Shape              Param #
ConvNet2D                                [1, 23]                   --
├─Spectrogram: 1-1                       [1, 2, 513, 884]          --
├─Conv2d: 1-2                            [1, 16, 253, 438]         2,608
├─MaxPool2d: 1-3                         [1, 16, 126, 219]         --
├─Conv2d: 1-4                            [1, 32, 120, 213]         25,120
├─MaxPool2d: 1-5                         [1, 32, 60, 106]          --
├─Conv2d: 1-6                            [1, 64, 56, 102]          51,264
├─MaxPool2d: 1-7                         [1, 64, 28, 51]           --
├─Conv2d: 1-8                            [1, 128, 24, 47]          204,928
├─MaxPool2d: 1-9                         [1, 128, 12, 23]          --
├─Conv2d: 1-10                           [1, 128, 4, 10]           409,728
├─MaxPool2d: 1-11                        [1, 128, 2, 5]            --
├─Linear: 1-12                           [1, 128]               

In [83]:
N_EPOCHS = 10
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
with tqdm(range(N_EPOCHS)) as pbar:
    for epoch in pbar:
        for (waveform, _, _, speaker_id, _) in train_loader:
            waveform = waveform.to(device)
            waveform = torch.cat([waveform, waveform], dim=1)
            speaker_one_hot = (torch.stack([onehot_speaker(i) for i in speaker_id]))
            speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
            speaker_one_hot = speaker_one_hot.to(device)
            optimizer.zero_grad()
            output = net(waveform)
            loss = criterion(output, speaker_id)
            loss.backward()
            optimizer.step()
            pbar.set_description(f"Loss: {loss.item():.4f}")

  0%|          | 0/10 [00:00<?, ?it/s]

Loss: 0.0232: 100%|██████████| 10/10 [12:46<00:00, 76.66s/it]


100%|██████████| 84/84 [00:06<00:00, 12.24it/s]

Accuracy: 93.89%


In [87]:
net_ds2 = ConvNet2D(nSpeakers).to(device)
N_EPOCHS = 10
optimizerds2 = torch.optim.Adam(net_ds2.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()
with tqdm(range(N_EPOCHS)) as pbar:
    for epoch in pbar:
        for (waveform, _, _, speaker_id, _) in train_ds2_loader:
            waveform = waveform.to(device)
            speaker_one_hot = (torch.stack([onehot_speaker(i) for i in speaker_id]))
            speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
            speaker_one_hot = speaker_one_hot.to(device)
            optimizerds2.zero_grad()
            output = net_ds2(waveform)
            loss = criterion(output, speaker_id)
            loss.backward()
            optimizerds2.step()
            pbar.set_description(f"Loss: {loss.item():.4f}")

Loss: 0.2234: 100%|██████████| 10/10 [13:38<00:00, 81.88s/it]


In [94]:
# save the models

torch.save(net.state_dict(), "convnet2d.pt")
torch.save(net_ds2.state_dict(), "convnet2d_ds2.pt")

In [95]:
accuracy = 0 
with torch.no_grad():
    for (waveform, _, _, speaker_id, _) in tqdm(test_loader):
        waveform = waveform.to(device)
        waveform = torch.cat([waveform, waveform], dim=1)
        speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
        output = net(waveform)
        accuracy += (output.argmax(1) == speaker_id).sum().item()
print(f"Accuracy: {accuracy / len(test_loader.dataset) * 100:.2f}%")

  0%|          | 0/84 [00:00<?, ?it/s]

100%|██████████| 84/84 [00:08<00:00,  9.84it/s]

Accuracy: 93.67%


In [96]:
accuracy = 0 
with torch.no_grad():
    for (waveform, _, _, speaker_id, _) in tqdm(test_ds2_loader):
        waveform = waveform.to(device)
        speaker_id = torch.tensor([speaker_list.index(i) for i in speaker_id]).to(device)
        output = net_ds2(waveform)
        accuracy += (output.argmax(1) == speaker_id).sum().item()
print(f"Accuracy: {accuracy / len(test_ds2_loader.dataset) * 100:.2f}%")

100%|██████████| 84/84 [00:10<00:00,  8.26it/s]

Accuracy: 89.05%
